# Radiomic preprocessing — Second-Progression target

## What this notebook produces

A single wide table `Processed/radiomic_features.csv` with **one row per
eligible patient** and **22 columns**:

```
Patient_ID,
vol_NCR, vol_SNFH, vol_ET, vol_RC                 # volumes per region (mm³)
t1c_mean_NCR, t1c_mean_SNFH, t1c_mean_ET, t1c_mean_RC,
t1n_mean_NCR, t1n_mean_SNFH, t1n_mean_ET, t1n_mean_RC,
t2f_mean_NCR, t2f_mean_SNFH, t2f_mean_ET, t2f_mean_RC,
t2w_mean_NCR, t2w_mean_SNFH, t2w_mean_ET, t2w_mean_RC,    # mean intensity per region × MRI modality
n_pre_landmark_scans,                              # how many seg rows passed gating
days_since_baseline                                # day of the snapshot relative to the patient's baseline
```

This is what Exp(iii) — *Molecular + Treatment + Timepoints + Radiomic*
— consumes inside the Mistral RAG prompt.

## Pipeline at a glance

```
Step 1  Load 4-sheet segmentation_volumes.xlsx, tag region.
Step 2  Concatenate, sort, attach Timepoint_idx via cumcount per (PID, Region).
Step 3  Map Timepoint_idx → Day_from_diag using the clinical sheet.
Step 4  Restrict to eligible cohort, attach y / Landmark_day / Death_day.
Step 5  Apply temporal gates (drop post-landmark and post-death rows).
Step 6  Per (Patient_ID × Region) keep ONLY the latest pre-landmark row.
Step 7  Pivot to wide: one row per Patient_ID, one column per (region × feature).
Step 8  Re-index to ALL eligible patients (NaN-fill those with no usable row).
Step 9  Write radiomic_features.csv  +  drop log  +  summary JSON.
```

## Why `latest pre-landmark snapshot per region` (not first)?

For Exp(iii) we want the **most recent radiomic view** of the tumour at
the moment the model is asked to predict.  Among rows with
`Day_from_diag < Landmark_day`, the latest one is the most relevant.

Patients with no recorded scan day are not used for radiomic features:
without a scan day we cannot prove that the image was acquired before
`Landmark_day`. Those rows are logged as `no-day` and the patient keeps
NaN radiomic values unless another proven pre-landmark scan exists.


In [ ]:
import sys, json
sys.path.insert(0, ".")
import pandas as pd, numpy as np
from pathlib import Path
from _landmark import (load_raw, eligible_cohort, attach_y_and_landmark, MRI_DAY_COLS)

PROC = Path("Processed"); PROC.mkdir(exist_ok=True)


## Step 1 — Load all four sheets and tag with region acronym


In [ ]:
SEG_PATH = Path("Raw/segmentation_volumes.xlsx")
SHEETS = {
    "Necrotic Tumor Core (Label1)":   "NCR",
    "Tumor Infiltration and Edema":   "SNFH",
    "Enhancing Tumor Core (Label3)":  "ET",
    "Resection Cavity (Label4)":      "RC",
}
frames = []
for sheet, region in SHEETS.items():
    df = pd.read_excel(SEG_PATH, sheet_name=sheet)
    df["Region"] = region
    frames.append(df)
seg = pd.concat(frames, ignore_index=True).rename(columns={"Patient ID": "Patient_ID"})
print(f"  total seg rows : {len(seg)}")
print(f"  per-region rows: {seg['Region'].value_counts().to_dict()}")


## Step 2 — Sort and assign Timepoint_idx via cumcount

Within each (Patient_ID × Region) group we assume rows are in
chronological order and label them 1, 2, 3, …


In [ ]:
seg = seg.sort_values(["Patient_ID","Region"]).reset_index(drop=True)
seg["Timepoint_idx"] = seg.groupby(["Patient_ID","Region"]).cumcount() + 1
print(f"  max Timepoint_idx observed : {seg['Timepoint_idx'].max()}")


## Step 3 — Map Timepoint_idx → Day_from_diag

The clinical sheet stores up to 6 MRI days per patient. We look up
day = `MRI_DAY_COLS[Timepoint_idx - 1]`.


In [ ]:
raw = load_raw()
day_map = raw.set_index("Patient_ID")[MRI_DAY_COLS].copy()
day_map.columns = list(range(1, len(MRI_DAY_COLS)+1))

def day_for(pid, idx):
    if pid in day_map.index and idx in day_map.columns:
        v = day_map.at[pid, idx]
        return float(v) if pd.notna(v) else np.nan
    return np.nan

seg["Day_from_diag"] = [day_for(p, t) for p, t in zip(seg["Patient_ID"], seg["Timepoint_idx"])]
print(f"  rows with Day_from_diag mapped : {seg['Day_from_diag'].notna().sum()} / {len(seg)}")


## Step 4 — Eligible-cohort filter + attach y / Landmark_day / Death_day


In [ ]:
elig = attach_y_and_landmark(eligible_cohort(raw))
elig_map = elig.set_index("Patient_ID")[["y","Landmark_day","Death_day"]]
seg = seg.merge(elig_map, left_on="Patient_ID", right_index=True, how="inner")
print(f"  rows kept (eligible) : {len(seg)}")
print(f"  unique eligible pids : {seg['Patient_ID'].nunique()} / {len(elig)}")


## Step 5 — Temporal-gate: drop post-landmark + post-death rows

Each dropped row is added to `radiomic_drop_log.csv`.


In [ ]:
drop_log = []

# Mark each row's gate
def gate(r):
    if pd.isna(r['Day_from_diag']):                      return "no-day"
    if r['Day_from_diag'] >= r['Landmark_day']:          return "post-landmark"
    if pd.notna(r['Death_day']) and r['Day_from_diag'] > r['Death_day']:
        return "post-death"
    return "pre-landmark"
seg['gate'] = seg.apply(gate, axis=1)

print("Gate counts (eligible cohort):")
print(seg['gate'].value_counts().to_string())
print()

# Log all non-pre-landmark rows
for g in ("post-landmark", "post-death", "no-day"):
    s = seg[seg['gate']==g]
    for _, r in s.iterrows():
        drop_log.append({"Patient_ID": r['Patient_ID'], "Region": r['Region'],
                         "Timepoint_idx": int(r['Timepoint_idx']),
                         "Day_from_diag": r['Day_from_diag'],
                         "Landmark_day":  r['Landmark_day'],
                         "Death_day":     r['Death_day'], "reason": g})

# Keep only proven pre-landmark rows for the snapshot.
# Rows with no recorded scan day are excluded because their timing cannot
# be verified against Landmark_day.
pre = seg[seg['gate']=='pre-landmark'].copy()
print(f"  pre-landmark rows : {len(pre)}")
print(f"  no-day rows dropped: {(seg['gate']=='no-day').sum()}")


## Step 6 — Per (Patient_ID × Region), keep the latest pre-landmark row

If a patient has no proven pre-landmark row for a region, that region
remains missing. We do not fall back to no-day rows.


In [ ]:
latest_pre = (pre.sort_values(['Patient_ID','Region','Day_from_diag'])
                  .groupby(['Patient_ID','Region']).tail(1)
                  .reset_index(drop=True))
print(f"  latest-pre snapshots: {len(latest_pre)}")

snapshot = latest_pre.copy()

print(f"  snapshots retained         : {len(snapshot)}")
print(f"  unique patients in snapshot: {snapshot['Patient_ID'].nunique()}")
print(f"  unique (pt × region)       : {snapshot[['Patient_ID','Region']].drop_duplicates().shape[0]}")


## Step 7 — Pivot to wide format

One row per `Patient_ID`, one column per `(region × feature)` pair.


In [ ]:
NUM_FEATS = ["Volume (mm^3)",
             "Image mean (brain_t1c)","Image mean (brain_t1n)",
             "Image mean (brain_t2f)","Image mean (brain_t2w)"]
SHORT = {"Volume (mm^3)": "vol",
         "Image mean (brain_t1c)": "t1c_mean",
         "Image mean (brain_t1n)": "t1n_mean",
         "Image mean (brain_t2f)": "t2f_mean",
         "Image mean (brain_t2w)": "t2w_mean"}

wide = snapshot.pivot_table(index="Patient_ID", columns="Region",
                             values=NUM_FEATS, aggfunc="first")
wide.columns = [f"{SHORT[feat]}_{region}" for feat, region in wide.columns]
print(f"  wide shape after pivot : {wide.shape}")


## Step 8 — Attach metadata + reindex to ALL eligible patients


In [ ]:
meta = (snapshot.groupby('Patient_ID')
                .agg(n_pre_landmark_scans = ('gate', lambda s: int((s=='pre-landmark').sum())),
                     latest_scan_day      = ('Day_from_diag', 'max'))
                .reset_index())
elig_meta = elig[['Patient_ID','Landmark_day']].merge(meta, on='Patient_ID', how='left')
elig_meta['days_since_baseline'] = (elig_meta['latest_scan_day']).round().astype('Int64')

wide = wide.merge(elig_meta.set_index('Patient_ID')[['n_pre_landmark_scans','latest_scan_day']],
                   left_index=True, right_index=True, how='left')
wide = wide.reindex(elig['Patient_ID']).reset_index()
print(f"  final shape : {wide.shape}")
print(f"  patients with NO usable pre-landmark scan: {wide['n_pre_landmark_scans'].fillna(0).eq(0).sum()}")


## Step 9 — Write artefacts


In [ ]:
wide.to_csv(PROC / "radiomic_features.csv", index=False)
print(f"  wrote {PROC/'radiomic_features.csv'}  ({len(wide)} rows × {wide.shape[1]} cols)")

drop_df = pd.DataFrame(drop_log)
if not drop_df.empty:
    drop_df.to_csv(PROC / "radiomic_drop_log.csv", index=False)
    print(f"  wrote {PROC/'radiomic_drop_log.csv'}  ({len(drop_df)} dropped/fallback rows)")

summary = {
    "n_eligible":               int(len(elig)),
    "n_with_any_pre_landmark":  int((wide['n_pre_landmark_scans'].fillna(0) > 0).sum()),
    "n_zero_pre_landmark":      int((wide['n_pre_landmark_scans'].fillna(0) == 0).sum()),
    "median_n_pre_landmark":    float(wide['n_pre_landmark_scans'].median(skipna=True) or 0.0),
    "median_latest_scan_day":   float(wide['latest_scan_day'].median(skipna=True) or 0.0),
    "feature_columns":          [c for c in wide.columns if c not in
                                 ('Patient_ID','n_pre_landmark_scans','latest_scan_day')],
    "drop_counts": (drop_df['reason'].value_counts().to_dict() if not drop_df.empty else {}),
}
with open(PROC / "radiomic_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print()
print(json.dumps(summary, indent=2))


## Sanity preview


In [ ]:
print("Patients per (col-availability bucket):")
print((wide['n_pre_landmark_scans'].fillna(0).astype(int)
        .value_counts().sort_index().to_string()))
print()
display(wide.head(8))
